# Inspect test_plant-0

This notebook reads the mesh file `test_plant-0.obj` and the same-name `test_plant-0.npz`, then shows their structure and sample contents.


In [3]:
from pathlib import Path
import zipfile

import numpy as np

OBJ_PATH = Path(r"C:\Users\sdjkl\CG\VRBioTalk\test_plants\test_plant-0.obj")
NPZ_PATH = OBJ_PATH.with_suffix(".npz")

print(f"OBJ exists: {OBJ_PATH.exists()} -> {OBJ_PATH}")
print(f"NPZ exists: {NPZ_PATH.exists()} -> {NPZ_PATH}")


OBJ exists: True -> C:\Users\sdjkl\CG\VRBioTalk\test_plants\test_plant-0.obj
NPZ exists: True -> C:\Users\sdjkl\CG\VRBioTalk\test_plants\test_plant-0.npz


In [4]:
def read_head(path: Path, num_lines: int = 20):
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        return [next(f).rstrip("\n") for _ in range(num_lines)]

obj_head = read_head(OBJ_PATH, num_lines=20)
print("First 20 lines of OBJ:")
for line in obj_head:
    print(line)


First 20 lines of OBJ:
# OBJ file generated by PlantModel
v -0.002756047062575817 -0.021805809810757637 0.0762578472495079
v -0.0021435923408716917 -0.01696007512509823 0.0762578472495079
v -0.0015311372699216008 -0.012114339508116245 0.0762578472495079
v -0.0009186823153868318 -0.0072686029598116875 0.0762578472495079
v -0.00030622744816355407 -0.0024228678084909916 0.0762578472495079
v 0.00030622744816355407 0.0024228678084909916 0.0762578472495079
v 0.0009186823153868318 0.0072686029598116875 0.0762578472495079
v 0.0015311372699216008 0.012114339508116245 0.0762578472495079
v 0.0021435923408716917 0.01696007512509823 0.0762578472495079
v 0.002756047062575817 0.021805809810757637 0.0762578472495079
v 0.011136304587125778 -0.02350030280649662 0.09290710091590881
v 0.01174706220626831 -0.01866799220442772 0.09290710091590881
v 0.012357821688055992 -0.013835676945745945 0.09290710091590881
v 0.012968579307198524 -0.009003364481031895 0.09290710091590881
v 0.013579338788986206 -0.0041710

In [5]:
vertices = []
vertex_normals = []
texcoords = []
faces = []

with OBJ_PATH.open("r", encoding="utf-8", errors="ignore") as f:
    for raw_line in f:
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("v "):
            vertices.append([float(v) for v in line.split()[1:4]])
        elif line.startswith("vn "):
            vertex_normals.append([float(v) for v in line.split()[1:4]])
        elif line.startswith("vt "):
            texcoords.append([float(v) for v in line.split()[1:]])
        elif line.startswith("f "):
            faces.append(line.split()[1:])

vertices_np = np.asarray(vertices, dtype=np.float32)

obj_summary = {
    "vertex_count": len(vertices),
    "face_count": len(faces),
    "normal_count": len(vertex_normals),
    "texcoord_count": len(texcoords),
    "bbox_min": vertices_np.min(axis=0).tolist() if len(vertices_np) else None,
    "bbox_max": vertices_np.max(axis=0).tolist() if len(vertices_np) else None,
    "first_5_vertices": vertices_np[:5].tolist(),
    "first_5_faces": faces[:5],
}

obj_summary


{'vertex_count': 3200,
 'face_count': 5472,
 'normal_count': 5472,
 'texcoord_count': 0,
 'bbox_min': [-0.2984178960323334, -0.15736830234527588, 0.04003637284040451],
 'bbox_max': [0.29881027340888977, 0.1051165834069252, 0.7138307094573975],
 'first_5_vertices': [[-0.002756047062575817,
   -0.021805809810757637,
   0.0762578472495079],
  [-0.0021435923408716917, -0.01696007512509823, 0.0762578472495079],
  [-0.0015311372699216008, -0.012114339508116245, 0.0762578472495079],
  [-0.0009186823153868318, -0.0072686029598116875, 0.0762578472495079],
  [-0.00030622744816355407, -0.0024228678084909916, 0.0762578472495079]],
 'first_5_faces': [['1//1', '12//12', '2//2'],
  ['1//1', '11//11', '12//12'],
  ['2//2', '13//13', '3//3'],
  ['2//2', '12//12', '13//13'],
  ['3//3', '14//14', '4//4']]}

In [6]:
with zipfile.ZipFile(NPZ_PATH, "r") as zf:
    archive_entries = [(info.filename, info.file_size) for info in zf.infolist()]

print("Raw npz entries:")
for name, size in archive_entries:
    print(f"  {name:20s} {size} bytes")

npz_data = np.load(NPZ_PATH, allow_pickle=True)
npz_summary = {}
for key in npz_data.files:
    arr = npz_data[key]
    preview = arr.reshape(-1)[:10].tolist() if arr.size else []
    npz_summary[key] = {
        "shape": arr.shape,
        "dtype": str(arr.dtype),
        "preview": preview,
    }

npz_summary


Raw npz entries:
  vertices.npy         38528 bytes
  faces.npy            131456 bytes
  normals.npy          65792 bytes
  instance_labels.npy  43904 bytes


{'vertices': {'shape': (3200, 3),
  'dtype': 'float32',
  'preview': [-0.002756047062575817,
   -0.021805809810757637,
   0.0762578472495079,
   -0.0021435923408716917,
   -0.01696007512509823,
   0.0762578472495079,
   -0.0015311372699216008,
   -0.012114339508116245,
   0.0762578472495079,
   -0.0009186823153868318]},
 'faces': {'shape': (5472, 3),
  'dtype': 'int64',
  'preview': [0, 11, 1, 0, 10, 11, 1, 12, 2, 1]},
 'normals': {'shape': (5472, 3),
  'dtype': 'float32',
  'preview': [-0.7594410181045532,
   0.09598612785339355,
   0.6434563994407654,
   -0.7594410181045532,
   0.09598615765571594,
   0.6434563398361206,
   -0.7594410181045532,
   0.09598612785339355,
   0.6434563398361206,
   -0.7594410181045532]},
 'instance_labels': {'shape': (5472,),
  'dtype': 'int64',
  'preview': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}}

In [7]:
npz_vertices = npz_data["vertices"]
npz_faces = npz_data["faces"]
npz_normals = npz_data["normals"]
npz_instance_labels = npz_data["instance_labels"]

comparison = {
    "obj_vertex_count": len(vertices),
    "npz_vertex_count": int(npz_vertices.shape[0]),
    "obj_face_count": len(faces),
    "npz_face_count": int(npz_faces.shape[0]),
    "vertex_count_match": len(vertices) == int(npz_vertices.shape[0]),
    "face_count_match": len(faces) == int(npz_faces.shape[0]),
    "first_5_npz_vertices": npz_vertices[:5].tolist(),
    "first_5_npz_faces": npz_faces[:5].tolist(),
    "first_5_npz_normals": npz_normals[:5].tolist(),
    "first_10_instance_labels": npz_instance_labels[:10].tolist(),
}

comparison


{'obj_vertex_count': 3200,
 'npz_vertex_count': 3200,
 'obj_face_count': 5472,
 'npz_face_count': 5472,
 'vertex_count_match': True,
 'face_count_match': True,
 'first_5_npz_vertices': [[-0.002756047062575817,
   -0.021805809810757637,
   0.0762578472495079],
  [-0.0021435923408716917, -0.01696007512509823, 0.0762578472495079],
  [-0.0015311372699216008, -0.012114339508116245, 0.0762578472495079],
  [-0.0009186823153868318, -0.0072686029598116875, 0.0762578472495079],
  [-0.00030622744816355407, -0.0024228678084909916, 0.0762578472495079]],
 'first_5_npz_faces': [[0, 11, 1],
  [0, 10, 11],
  [1, 12, 2],
  [1, 11, 12],
  [2, 13, 3]],
 'first_5_npz_normals': [[-0.7594410181045532,
   0.09598612785339355,
   0.6434563994407654],
  [-0.7594410181045532, 0.09598615765571594, 0.6434563398361206],
  [-0.7594410181045532, 0.09598612785339355, 0.6434563398361206],
  [-0.7594410181045532, 0.09598615765571594, 0.6434563994407654],
  [-0.7594410181045532, 0.09598612785339355, 0.6434563994407654]],

In [ ]:
try:
    import matplotlib.pyplot as plt

    sample_stride = max(1, len(npz_vertices) // 2000)
    sampled = npz_vertices[::sample_stride]

    fig = plt.figure(figsize=(7, 7))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(sampled[:, 0], sampled[:, 1], sampled[:, 2], s=2)
    ax.set_title("Sampled vertices from test_plant-0")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    plt.show()
except ModuleNotFoundError:
    print("matplotlib is not installed in this environment, skip plotting.")
